# EVA-02 Two-Stage Baseline: 448 px → 672 px

This notebook trains an EVA-02 image classifier in two stages on the ISIC 2019 dataset. Stage 1 uses 448 × 448 inputs; Stage 2 initializes a new model at 672 × 672 and transfers the Stage 1 checkpoint with positional-embedding interpolation. The notebook then evaluates the selected checkpoint with test-time augmentation, exports metrics and image features, and provides optional Grad-CAM/EigenCAM visualizations.

## Before running

- Use a CUDA-enabled runtime with sufficient GPU memory. The 672 × 672 stage is memory-intensive.
- The default workflow is designed for Google Colab and attempts to mount Google Drive.
- The split directory must contain `train_split.csv`, `val_split.csv`, and `test_split.csv`. Each CSV must include `image` and `class_name` columns. `image` should contain the image identifier without the `.jpg` extension; `class_name` must match one of the eight class labels defined in `CFG.CLASS_NAMES`.
- The image archive is downloaded from the ISIC source if it is not already present.
- No dataset images, split CSV files, checkpoints, predictions, feature arrays, or generated CAM figures are included in this repository.
- For an exact reproduction, confirm that the model name, split CSVs, hyperparameters, library versions, and preprocessing match the final manuscript.

> **Reproducibility note:** The training and evaluation code below is retained from the research workflow and has been cleaned for repository sharing. It has not been executed end-to-end in this environment, so verify it in a fresh Colab runtime before citing it as a fully validated reproduction package.


## 1. Download and extract the ISIC 2019 image archive

This cell downloads the official ISIC 2019 training-image ZIP archive into the runtime and extracts it under `/content/ISIC_2019_Training_Input`. It skips download or extraction when the corresponding file/directory already exists.

**Output:** the extracted image directory. The dataset itself is not stored in this GitHub repository.


In [ ]:
# ============================================================
# Download and extract the ISIC 2019 training images from the official archive.
# ============================================================
import os, zipfile, requests
from tqdm.auto import tqdm

URL = "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip"
ZIP_PATH = "/content/ISIC_2019_Training_Input.zip"
EXTRACT_DIR = "/content"  # The archive is expected to contain the 'ISIC_2019_Training_Input' directory.

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Download the archive and display byte-level progress.
if not os.path.exists(ZIP_PATH):
    print("Downloading dataset:", URL)
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        pbar = tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc="Downloading")
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
        pbar.close()
else:
    print("Archive already exists:", ZIP_PATH)

# Extract the archive and display file-level progress.
target_path = os.path.join(EXTRACT_DIR, "ISIC_2019_Training_Input")
if not os.path.exists(target_path):
    print("Extracting archive to:", EXTRACT_DIR)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        members = zf.infolist()
        pbar = tqdm(total=len(members), desc="Extracting", unit="file")
        for m in members:
            zf.extract(m, EXTRACT_DIR)
            pbar.update(1)
        pbar.close()
else:
    print("Target directory already exists:", target_path)

# -------- Son durum --------
print("\nDataset is ready:", target_path)
print("First few files:", sorted(os.listdir(target_path))[:5])


## 2. Train, fine-tune, evaluate, and export EVA-02 features

This cell contains the main experiment pipeline:

1. Install the principal runtime dependencies and import libraries.
2. Set the random seed and hardware options.
3. Load the train/validation/test split CSV files and map class labels.
4. Train EVA-02 at 448 px and select the best checkpoint by validation Macro-F1.
5. Initialize a 672 px model, interpolate positional embeddings, and fine-tune it.
6. Evaluate the selected checkpoint using three-view test-time augmentation (original, horizontal flip, vertical flip).
7. Save classification reports, confusion matrices, ROC curves, metrics, a run manifest, and train/validation/test feature arrays.

**Configuration:** adjust the `CFG` class or set `EVA02_OUTPUT_DIR`, `SPLIT_CSV_DIR`, and `ISIC_IMAGE_DIR` environment variables. Keep test data out of model selection and hyperparameter tuning.

**Storage note:** the run creates large checkpoint and feature files outside the repository. Do not commit these generated artifacts by default.


In [ ]:
# ======================================================================
# EVA-02 two-stage training and evaluation pipeline (448 px → 672 px).
# - Stage 1: train and save the model at 448 px.
# - Stage 2: initialize a new 672 px model and interpolate positional embeddings from Stage 1.
# - Checkpoint helpers support both compiled and non-compiled models.
# - Export metrics, figures, and features for the train/validation/test splits.
# ======================================================================

print("Installing required packages...")
!pip -q install timm albumentations

import os, json, time, random, warnings, zipfile
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch, timm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast

from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             matthews_corrcoef, roc_auc_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns

from timm.models.vision_transformer import resize_pos_embed  # pos_embed interp

from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception:
    print("Google Drive is already mounted, or this notebook is running outside Colab.")

# ------------------- Repro: sabit tohumlar -------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

# ------------------- Hardware and performance settings -------------------
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ------------------- CFG -------------------
class CFG:
    MODEL_NAME = 'eva02_base_patch14_448.mim_in22k_ft_in1k'
    STAGE1_IMG = 448
    STAGE2_IMG = 672
    NUM_CLASSES = 8
    CLASS_NAMES = ['AK','BCC','BKL','DF','MEL','NV','SCC','VASC']
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Persistent output directory. Override with EVA02_OUTPUT_DIR if needed.
    ROOT = os.environ.get("EVA02_OUTPUT_DIR", "/content/drive/MyDrive/skin_lesion_research/eva02")
    # This directory must contain train_split.csv, val_split.csv, and test_split.csv.
    # Each CSV must include the columns: image and class_name.
    DATASET_CSV_DIR = os.environ.get("SPLIT_CSV_DIR", "/content/drive/MyDrive/skin_lesion_research/splits")
    # Set ISIC_IMAGE_DIR if the extracted image directory is stored elsewhere.
    IMAGE_DIR = os.environ.get("ISIC_IMAGE_DIR", "/content/ISIC_2019_Training_Input")

    # Reduce these values for a smoke test; use the final study settings for reproduction.
    EPOCHS = 200          # S1 (448)
    FT_EPOCHS = 100       # S2 (672)

    LEARNING_RATE = 1e-5
    FT_LEARNING_RATE = 5e-6
    WEIGHT_DECAY = 1e-3

    PATIENCE = 20
    FT_PATIENCE = 20

    BS_STAGE1 = 32      # 448
    BS_STAGE2 = 16       # 672 px; reduce this value if GPU memory is insufficient.

# Output directories
SUBDIRS = {
    "ckpt": os.path.join(CFG.ROOT, "checkpoints"),
    "fig":  os.path.join(CFG.ROOT, "figures"),
    "met":  os.path.join(CFG.ROOT, "metrics"),
    "feat": os.path.join(CFG.ROOT, "features"),
    "logs": os.path.join(CFG.ROOT, "logs"),
}
for d in SUBDIRS.values(): os.makedirs(d, exist_ok=True)

print(f"Cihaz: {CFG.DEVICE} | Model: {CFG.MODEL_NAME} | Stages: {CFG.STAGE1_IMG} → {CFG.STAGE2_IMG}")
print(f"Output directory: {CFG.ROOT}")

# ------------------- Download ISIC 2019 images if needed -------------------
if not os.path.exists(CFG.IMAGE_DIR):
    print(f"Image directory not found. Downloading to: {CFG.IMAGE_DIR}")
    URL = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip"
    ZIP_PATH = "/content/ISIC_2019_Training_Input.zip"
    os.system(f"wget -q --show-progress -O {ZIP_PATH} {URL}")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    os.remove(ZIP_PATH)
    print("ISIC 2019 images are ready.")
else:
    print("Image directory already exists.")

# ------------------- Image transformations -------------------
import albumentations as A
from albumentations.pytorch import ToTensorV2
def make_tfms(img_size):
    train = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ColorJitter(0.1,0.1,0.1,0.1, p=0.3),
        A.CoarseDropout(max_holes=8, max_height=int(img_size*0.1),
                        max_width=int(img_size*0.1), p=0.3),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])
    val = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])
    return train, val

# ------------------- Dataset wrapper -------------------
class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'] + ".jpg")
        image = np.array(Image.open(img_path).convert('RGB'))
        label = row['label']
        if self.transforms: image = self.transforms(image=image)['image']
        return image, label

# ------------------- Split CSV files and class labels -------------------
df_train = pd.read_csv(os.path.join(CFG.DATASET_CSV_DIR, "train_split.csv"))
df_val   = pd.read_csv(os.path.join(CFG.DATASET_CSV_DIR, "val_split.csv"))
df_test  = pd.read_csv(os.path.join(CFG.DATASET_CSV_DIR, "test_split.csv"))
class_map = {name: i for i, name in enumerate(CFG.CLASS_NAMES)}

for split_name, split_df in (("train", df_train), ("validation", df_val), ("test", df_test)):
    required_columns = {"image", "class_name"}
    missing_columns = required_columns - set(split_df.columns)
    if missing_columns:
        raise ValueError(f"The {split_name} split is missing required columns: {sorted(missing_columns)}")
    if split_df["class_name"].isna().any():
        raise ValueError(f"The {split_name} split contains missing class labels.")
    unknown_classes = sorted(set(split_df["class_name"].unique()) - set(class_map))
    if unknown_classes:
        raise ValueError(f"Unknown class labels in the {split_name} split: {unknown_classes}")
    split_df["label"] = split_df["class_name"].map(class_map).astype(int)

# ------------------- Model helpers -------------------
def build_model(img_size):
    try:
        m = timm.create_model(
            CFG.MODEL_NAME, pretrained=True, num_classes=CFG.NUM_CLASSES,
            img_size=img_size, fused_attn=True
        ).to(CFG.DEVICE)
    except TypeError:
        m = timm.create_model(
            CFG.MODEL_NAME, pretrained=True, num_classes=CFG.NUM_CLASSES,
            img_size=img_size
        ).to(CFG.DEVICE)
    m = m.to(memory_format=torch.channels_last)
    try:
        m = torch.compile(m, mode="max-autotune")
        print(f"torch.compile enabled for image size {img_size}.")
    except Exception as e:
        print(f"torch.compile could not be enabled for image size {img_size}:", e)
    return m

def _get_base(m):  # Return the original module when the model is compiled.
    return getattr(m, "_orig_mod", m)

def save_ckpt_safely(m, path):
    base = _get_base(m)
    torch.save(base.state_dict(), path)

def load_ckpt_with_pos_interp(model_dst, state_src):
    """Interpolate positional embeddings when transferring Stage 1 weights to Stage 2."""
    dst = model_dst.state_dict()
    new_state = {}
    for k, v in state_src.items():
        kk = k.replace("_orig_mod.", "")  # compile izlerini temizle
        if kk in dst and v.shape != dst[kk].shape and kk == "pos_embed":
            new_state[kk] = resize_pos_embed(v, dst[kk])
        else:
            new_state[kk] = v
    missing, unexpected = model_dst.load_state_dict(new_state, strict=False)
    if missing:   print("⚠️ Missing keys:", missing[:5], "...")
    if unexpected:print("⚠️ Unexpected keys:", unexpected[:5], "...")

# ------------------- Loss, optimizer, and scheduler -------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = (1-pt)**self.gamma * ce
        if self.alpha is not None:
            a = self.alpha.to(inputs.device)
            at = a.gather(0, targets.view(-1))
            loss = at * loss
        return loss.mean()

from sklearn.utils.class_weight import compute_class_weight
cls_w = compute_class_weight('balanced', classes=np.arange(CFG.NUM_CLASSES), y=df_train['label'].values)
criterion = FocalLoss(alpha=torch.tensor(cls_w, dtype=torch.float), gamma=2.0)

def make_optimizer(model, lr):
    try:
        return AdamW(model.parameters(), lr=lr, weight_decay=CFG.WEIGHT_DECAY, fused=True)
    except TypeError:
        return AdamW(model.parameters(), lr=lr, weight_decay=CFG.WEIGHT_DECAY)

# ------------------- Shared evaluation helpers -------------------
@torch.no_grad()
def evaluate(m, loader, criterion, return_probs=False):
    m.eval(); total=0.0; y_true=[]; y_pred=[]; y_prob=[]
    for imgs, labels in loader:
        imgs = imgs.to(CFG.DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(CFG.DEVICE, non_blocking=True)
        with autocast(dtype=torch.bfloat16):
            logits = m(imgs); loss = criterion(logits, labels)
        total += loss.item()
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax(1)
        y_true.extend(labels.cpu().numpy()); y_pred.extend(pred.cpu().numpy())
        if return_probs: y_prob.extend(probs.to(torch.float32).cpu().numpy())
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    out = (total / max(1, len(loader)), f1, np.array(y_true), np.array(y_pred))
    if return_probs: out = (*out, np.array(y_prob))
    return out

def save_report_and_cm(y_true, y_pred, split_tag):
    cr_txt = classification_report(y_true, y_pred, target_names=CFG.CLASS_NAMES, digits=4)
    with open(os.path.join(SUBDIRS["met"], f"{split_tag}_classification_report.txt"), "w") as f:
        f.write(cr_txt)
    print("\n" + "="*60 + f"\n{split_tag.upper()} — Classification Report\n" + "="*60)
    print(cr_txt)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(CFG.NUM_CLASSES))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    pd.DataFrame(cm, index=CFG.CLASS_NAMES, columns=CFG.CLASS_NAMES)\
      .to_csv(os.path.join(SUBDIRS["met"], f"{split_tag}_cm_raw.csv"))
    pd.DataFrame(cm_norm, index=CFG.CLASS_NAMES, columns=CFG.CLASS_NAMES)\
      .to_csv(os.path.join(SUBDIRS["met"], f"{split_tag}_cm_norm.csv"))

    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
                xticklabels=CFG.CLASS_NAMES, yticklabels=CFG.CLASS_NAMES)
    plt.title(f"{split_tag.upper()} — Confusion Matrix (raw)")
    plt.xlabel("Pred"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(os.path.join(SUBDIRS["fig"], f"{split_tag}_cm_raw.png"), dpi=200)
    plt.show()

    plt.figure(figsize=(10,8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='viridis',
                xticklabels=CFG.CLASS_NAMES, yticklabels=CFG.CLASS_NAMES)
    plt.title(f"{split_tag.upper()} — Confusion Matrix (normalized)")
    plt.xlabel("Pred"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(os.path.join(SUBDIRS["fig"], f"{split_tag}_cm_norm.png"), dpi=200)
    plt.show()

def save_roc_auc(y_true, y_prob, split_tag):
    y_true_oh = label_binarize(y_true, classes=np.arange(CFG.NUM_CLASSES))
    try:
        auc_macro = roc_auc_score(y_true_oh, y_prob, average="macro")
        auc_micro = roc_auc_score(y_true_oh, y_prob, average="micro")
    except Exception:
        auc_macro, auc_micro = np.nan, np.nan

    fpr, tpr, roc_auc = {}, {}, {}
    for i, cls in enumerate(CFG.CLASS_NAMES):
        y_i = y_true_oh[:, i]
        if y_i.sum() == 0 or (1 - y_i).sum() == 0:
            fpr[i], tpr[i], roc_auc[i] = np.array([0,1]), np.array([0,1]), np.nan
            continue
        fpr[i], tpr[i], _ = roc_curve(y_i, y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(9,8))
    try:
        y_true_r = y_true_oh.ravel(); y_prob_r = y_prob.ravel()
        fpr_micro, tpr_micro, _ = roc_curve(y_true_r, y_prob_r)
        auc_micro_plot = auc(fpr_micro, tpr_micro)
        plt.plot(fpr_micro, tpr_micro, label=f"micro-average ROC (AUC={auc_micro_plot:.3f})", lw=2)
    except Exception:
        pass
    for i, cls in enumerate(CFG.CLASS_NAMES):
        if not np.isnan(roc_auc[i]):
            plt.plot(fpr[i], tpr[i], lw=1, alpha=0.8, label=f"{cls} (AUC={roc_auc[i]:.3f})")
    plt.plot([0,1],[0,1], 'k--', lw=1)
    plt.xlim([0.0,1.0]); plt.ylim([0.0,1.05])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"{split_tag.upper()} — ROC Curves (class/micro)")
    plt.legend(loc="lower right", fontsize=8, ncol=2)
    plt.tight_layout(); plt.savefig(os.path.join(SUBDIRS["fig"], f"{split_tag}_roc.png"), dpi=200)
    plt.show()

    meta = {
        "auc_macro": float(auc_macro) if auc_macro==auc_macro else None,
        "auc_micro": float(auc_micro) if auc_micro==auc_micro else None,
        "auc_per_class": {CFG.CLASS_NAMES[i]: (None if np.isnan(roc_auc[i]) else float(roc_auc[i]))
                          for i in range(CFG.NUM_CLASSES)}
    }
    with open(os.path.join(SUBDIRS["met"], f"{split_tag}_auc.json"), "w") as f:
        json.dump(meta, f, indent=2)

def mcc_and_f1(y_true, y_pred, split_tag):
    mcc = matthews_corrcoef(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro", zero_division=0)
    with open(os.path.join(SUBDIRS["met"], f"{split_tag}_mcc_f1.json"), "w") as f:
        json.dump({"mcc": float(mcc), "macro_f1": float(f1m)}, f, indent=2)
    print(f"\n{split_tag.upper()} — MCC: {mcc:.4f} | Macro-F1: {f1m:.4f}")

# ------------------- Stage 1: training at 448 px -------------------
print("\n--- Stage 1: main training (448 px) ---")
model_s1 = build_model(CFG.STAGE1_IMG)
train_tfms_s1, val_tfms_s1 = make_tfms(CFG.STAGE1_IMG)
nw = max(2, min(8, os.cpu_count() or 2))
train_loader = DataLoader(ISICDataset(df_train, CFG.IMAGE_DIR, transforms=train_tfms_s1),
                          batch_size=CFG.BS_STAGE1, shuffle=True, num_workers=nw,
                          pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(ISICDataset(df_val, CFG.IMAGE_DIR, transforms=val_tfms_s1),
                          batch_size=CFG.BS_STAGE1*2, shuffle=False, num_workers=nw,
                          pin_memory=True, persistent_workers=True, prefetch_factor=4)

optimizer = make_optimizer(model_s1, CFG.LEARNING_RATE)
scheduler = CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

def train_one_epoch(m, loader, opt, crit, desc):
    m.train(); total=0.0
    for imgs, labels in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(CFG.DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(CFG.DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(dtype=torch.bfloat16):
            outputs = m(imgs); loss = crit(outputs, labels)
        loss.backward(); opt.step()
        total += loss.item()
    return total / max(1, len(loader))

best_main = os.path.join(SUBDIRS["ckpt"], "best_model_stage1_main.pth")
best_ft   = os.path.join(SUBDIRS["ckpt"], "best_model_stage2_finetuned.pth")

best_f1 = -1.0; pat = 0
for epoch in range(CFG.EPOCHS):
    tr_loss = train_one_epoch(model_s1, train_loader, optimizer, criterion, "Training S1")
    val_loss, val_f1, yv, yp = evaluate(model_s1, val_loader, criterion)
    scheduler.step()
    print(f"[S1] Epoch {epoch+1}/{CFG.EPOCHS} | Train {tr_loss:.4f} | Val {val_loss:.4f} | F1 {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1; pat = 0
        save_ckpt_safely(model_s1, best_main)
        print(f"Best Stage 1 validation Macro-F1: {best_f1:.4f} — {best_main}")
    else:
        pat += 1
        if pat >= CFG.PATIENCE:
            print("Early stopping triggered for Stage 1."); break

save_report_and_cm(yv, yp, "val_stage1")
mcc_and_f1(yv, yp, "val_stage1")

# ------------------- Stage 2: fine-tuning at 672 px -------------------
print("\n--- Stage 2: fine-tuning (672 px) ---")
model_s2 = build_model(CFG.STAGE2_IMG)  # yeni model 672
# Load the Stage 1 checkpoint and interpolate positional embeddings for the Stage 2 model.
state_s1 = torch.load(best_main, map_location=CFG.DEVICE)
load_ckpt_with_pos_interp(_get_base(model_s2), state_s1)

train_tfms_s2, val_tfms_s2 = make_tfms(CFG.STAGE2_IMG)
train_loader = DataLoader(ISICDataset(df_train, CFG.IMAGE_DIR, transforms=train_tfms_s2),
                          batch_size=CFG.BS_STAGE2, shuffle=True, num_workers=nw,
                          pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(ISICDataset(df_val, CFG.IMAGE_DIR, transforms=val_tfms_s2),
                          batch_size=CFG.BS_STAGE2*2, shuffle=False, num_workers=nw,
                          pin_memory=True, persistent_workers=True, prefetch_factor=4)

optimizer = make_optimizer(model_s2, CFG.FT_LEARNING_RATE)
scheduler = CosineAnnealingLR(optimizer, T_max=CFG.FT_EPOCHS, eta_min=1e-7)

best_ft_f1 = best_f1; pat = 0
for epoch in range(CFG.FT_EPOCHS):
    tr_loss = train_one_epoch(model_s2, train_loader, optimizer, criterion, "Training S2")
    val_loss, val_f1, yv, yp = evaluate(model_s2, val_loader, criterion)
    scheduler.step()
    print(f"[S2] FT Epoch {epoch+1}/{CFG.FT_EPOCHS} | Train {tr_loss:.4f} | Val {val_loss:.4f} | F1 {val_f1:.4f}")
    if val_f1 > best_ft_f1:
        best_ft_f1 = val_f1; pat = 0
        save_ckpt_safely(model_s2, best_ft)
        print(f"Best Stage 2 validation Macro-F1: {best_ft_f1:.4f} — {best_ft}")
    else:
        pat += 1
        if pat >= CFG.FT_PATIENCE:
            print("Early stopping triggered for Stage 2."); break

# ------------------- Final model and test-time augmentation (horizontal/vertical flips) -------------------
final_path = best_ft if os.path.exists(best_ft) else best_main
print(f"\nFinal checkpoint: {final_path}")

# Build an uncompiled 672 px model and load the selected checkpoint.
final_model = timm.create_model(CFG.MODEL_NAME, pretrained=False, num_classes=CFG.NUM_CLASSES, img_size=CFG.STAGE2_IMG).to(CFG.DEVICE)
# Load Stage 2 weights when available; otherwise interpolate the Stage 1 positional embeddings.
state_final = torch.load(final_path, map_location=CFG.DEVICE)
try:
    missing, unexpected = final_model.load_state_dict(state_final, strict=True)
except Exception:
    load_ckpt_with_pos_interp(final_model, state_final)
final_model = final_model.to(memory_format=torch.channels_last).eval()

@torch.no_grad()
def test_tta(m, loader):
    y_true=[]; y_pred=[]; y_prob=[]
    for imgs, labels in tqdm(loader, desc="TEST (TTA)", leave=False):
        x = imgs.to(CFG.DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with autocast(dtype=torch.bfloat16):
            o0 = m(x); o1 = m(torch.flip(x, dims=[3])); o2 = m(torch.flip(x, dims=[2]))
            p = (torch.softmax(o0,1)+torch.softmax(o1,1)+torch.softmax(o2,1))/3.0
            pred = p.argmax(1)
        y_true.extend(labels.numpy()); y_pred.extend(pred.cpu().numpy()); y_prob.extend(p.to(torch.float32).cpu().numpy())
    return np.array(y_true), np.array(y_pred), np.array(y_prob)

# Test data loader at 672 px.
test_tfms = A.Compose([A.Resize(CFG.STAGE2_IMG, CFG.STAGE2_IMG),
                       A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
                       ToTensorV2()])
test_loader = DataLoader(ISICDataset(df_test, CFG.IMAGE_DIR, transforms=test_tfms),
                         batch_size=CFG.BS_STAGE2*2, shuffle=False, num_workers=nw,
                         pin_memory=True, persistent_workers=True, prefetch_factor=4)

yt, yp, yprobs = test_tta(final_model, test_loader)
save_report_and_cm(yt, yp, "test_tta"); mcc_and_f1(yt, yp, "test_tta"); save_roc_auc(yt, yprobs, "test_tta")

# ------------------- Validation ROC/AUC -------------------
val_eval_loader = DataLoader(ISICDataset(df_val, CFG.IMAGE_DIR, transforms=test_tfms),
                             batch_size=CFG.BS_STAGE2*2, shuffle=False, num_workers=nw,
                             pin_memory=True, persistent_workers=True, prefetch_factor=4)
_, _, yv_final, yp_final, yprobs_val = evaluate(final_model, val_eval_loader, criterion, return_probs=True)
save_roc_auc(yv_final, yprobs_val, "val_final")

# ------------------- Feature export (train/validation/test, 672 px evaluation transforms) -------------------
@torch.no_grad()
def extract_features(m, loader, split_tag):
    m.eval()
    feats_list=[]; labels_list=[]
    for imgs, labels in tqdm(loader, desc=f"FEAT {split_tag}", leave=False):
        x = imgs.to(CFG.DEVICE, non_blocking=True, memory_format=torch.channels_last)
        try:
            f = m.forward_features(x)
            f = m.forward_head(f, pre_logits=True)
        except Exception as exc:
            raise RuntimeError(
                "EVA-02 feature extraction failed. Check the timm version and model API; "
                "the code will not silently substitute classifier logits for features."
            ) from exc
        feats_list.append(f.to(torch.float32).detach().cpu().numpy())
        labels_list.extend(labels.numpy())
    feats = np.concatenate(feats_list, axis=0)
    np.save(os.path.join(SUBDIRS["feat"], f"{split_tag}_features.npy"), feats)
    pd.DataFrame({"index": np.arange(len(labels_list)), "label": labels_list})\
      .to_csv(os.path.join(SUBDIRS["feat"], f"{split_tag}_meta.csv"), index=False)
    print(f"Saved {split_tag} features with shape: {feats.shape}")

train_eval_loader = DataLoader(ISICDataset(df_train, CFG.IMAGE_DIR, transforms=test_tfms),
                               batch_size=CFG.BS_STAGE2*2, shuffle=False, num_workers=nw,
                               pin_memory=True, persistent_workers=True, prefetch_factor=4)
extract_features(final_model, train_eval_loader, "train")
extract_features(final_model, val_eval_loader,   "val")
extract_features(final_model, test_loader,       "test")

# ------------------- Experiment manifest -------------------
manifest = {
    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_name": CFG.MODEL_NAME,
    "stage1_img": CFG.STAGE1_IMG,
    "stage2_img": CFG.STAGE2_IMG,
    "epochs_stage1": CFG.EPOCHS,
    "epochs_stage2": CFG.FT_EPOCHS,
    "batch_stage1": CFG.BS_STAGE1,
    "batch_stage2": CFG.BS_STAGE2,
    "optimizer": "AdamW",
    "lr_stage1": CFG.LEARNING_RATE,
    "lr_stage2": CFG.FT_LEARNING_RATE,
    "weight_decay": CFG.WEIGHT_DECAY,
    "classes": CFG.CLASS_NAMES,
    "paths": SUBDIRS
}
with open(os.path.join(SUBDIRS["logs"], "run_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print("\nThe 448→672 training and evaluation pipeline has finished. Outputs were saved.\n"
      f"Output directory: {CFG.ROOT}\n"
      f"   - Checkpoints: {SUBDIRS['ckpt']}\n"
      f"   - Metrics:     {SUBDIRS['met']}\n"
      f"   - Figures:     {SUBDIRS['fig']}\n"
      f"   - Features:    {SUBDIRS['feat']}")


## 3. Output precision and feature-export note

The original notebook contained a separate continuation/patch cell that redefined evaluation and feature-export helpers after the main training cell. The FP32 conversions needed before NumPy export have now been incorporated directly into the main pipeline, so the duplicated patch cell has been removed. This avoids repeating validation evaluation and feature extraction when the main cell completes successfully.


## 4. Optional single-image Grad-CAM/EigenCAM visualization

Run this section only after the main training cell has completed successfully and `final_model`, `CFG`, `A`, `ToTensorV2`, and `df_test` exist in memory. It selects up to `N` examples from the test split and saves one Grad-CAM or EigenCAM visualization per image.

The token reshaping function assumes one class token followed by square patch tokens. Verify this assumption for the exact EVA-02 implementation and `timm` version used in the final experiment.


In [ ]:
# ================== Grad-CAM for ViT/EVA-style models ==================
# Uses the current Grad-CAM API and can both save and display visualizations.
!pip -q install grad-cam --quiet

import os, cv2, numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage
from pytorch_grad_cam import GradCAM, EigenCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Output directory
OUT_DIR = os.path.join(SUBDIRS["fig"], "cam")
os.makedirs(OUT_DIR, exist_ok=True)
print("CAM output directory:", OUT_DIR)

# Evaluation transforms matching the final model's input resolution.
IMG_SZ = getattr(final_model, 'img_size', 672)
eval_tfms = A.Compose([
    A.Resize(IMG_SZ, IMG_SZ),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

# Convert a normalized image tensor back to the [0, 1] range for visualization.
def to_float01(img_tensor):
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    img = (img_tensor.cpu()*std + mean).clamp(0,1).permute(1,2,0).numpy()
    return img.astype(np.float32)  # HxWx3, float32 [0,1]

# Reshape ViT/EVA tokens from [B, N, C] to a 2D feature map [B, C, H, W].
def vit_reshape_transform(tensor):
    if tensor.ndim == 4:  # [B, C, H, W]
        return tensor
    if tensor.ndim != 3:  # [B, N, C] bekleriz
        raise ValueError(f"Unexpected tensor shape: {tuple(tensor.shape)}")
    B, N, C = tensor.shape
    patch_tokens = tensor[:, 1:, :]  # Remove the class token.
    S = int(np.floor(np.sqrt(patch_tokens.shape[1])))
    patch_tokens = patch_tokens[:, :S*S, :]
    H = W = S
    patch_tokens = patch_tokens.permute(0, 2, 1).contiguous().view(B, C, H, W)
    return patch_tokens

# Target layer: use the final block norm, falling back to the attention projection.
try:
    TARGET_LAYERS = [getattr(final_model, "blocks")[-1].norm1]
except Exception:
    TARGET_LAYERS = [getattr(getattr(final_model, "blocks")[-1].attn, "proj")]

@torch.no_grad()
def predict_logits(img_tensor):
    final_model.eval()
    x = img_tensor.unsqueeze(0).to(CFG.DEVICE, non_blocking=True, memory_format=torch.channels_last)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        logits = final_model(x)
        probs = torch.softmax(logits, 1)
    return logits, probs

def make_cam_for_one(image_path, true_label=None, topk=1, method="gradcam", show=True, save=True):
    # 1) Load the image and convert it to a tensor.
    image = np.array(PILImage.open(image_path).convert('RGB'))
    t = eval_tfms(image=image)["image"]  # (3,H,W)

    # 2) Tahmin
    _, probs = predict_logits(t)
    pred_idx = int(probs.argmax(1).item())
    topk_idx = probs.topk(topk, dim=1).indices[0].tolist()

    # 3) Select Grad-CAM or EigenCAM.
    cam_cls = GradCAM if method.lower() == "gradcam" else EigenCAM

    # 4) Compute the class activation map; reshape_transform is important for ViT/EVA tokens.
    with cam_cls(model=final_model, target_layers=TARGET_LAYERS,
                 reshape_transform=vit_reshape_transform) as cam:
        rgb = to_float01(t)  # [0,1]
        grayscale_cam = cam(
            input_tensor=t.unsqueeze(0).to(CFG.DEVICE),
            targets=[ClassifierOutputTarget(topk_idx[0])]
        )[0]  # (H,W)
        cam_img = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)

    # 5) Kaydet
    out_path = None
    if save:
        base = os.path.splitext(os.path.basename(image_path))[0]
        out_path = os.path.join(OUT_DIR, f"CAM_{base}_{method}.png")
        cv2.imwrite(out_path, cv2.cvtColor((cam_img*255).astype(np.uint8), cv2.COLOR_RGB2BGR))
        print(f"Saved visualization to: {out_path}")

    # 6) Display the visualization.
    if show:
        tl = CFG.CLASS_NAMES[true_label] if true_label is not None else "?"
        pdname = CFG.CLASS_NAMES[pred_idx]
        title = f"pred={pdname} | true={tl} | top1={CFG.CLASS_NAMES[topk_idx[0]]}"
        plt.figure(figsize=(6.2, 6))
        plt.imshow(cam_img); plt.title(title); plt.axis('off')
        plt.show()

    return out_path, cam_img

# ---------------- Generate visualizations for N samples from the test split ----------------
N = 12  # Adjust the number of samples as needed.
sample_df = df_test.sample(n=min(N, len(df_test)), random_state=42).reset_index(drop=True)
for _, row in sample_df.iterrows():
    img_path = os.path.join(CFG.IMAGE_DIR, row["image"] + ".jpg")
    make_cam_for_one(img_path, true_label=row["label"], topk=1, method="gradcam", show=True, save=True)

print("CAM generation completed. Output directory:", OUT_DIR)


## 5. Optional multi-panel CAM visualizations and mask-based activation summary

This section produces a 2 × 3 figure containing the original image, Grad-CAM and EigenCAM overlays, a Grad-CAM heatmap, and the second- and third-ranked class overlays. It can optionally compute activation averages inside and outside a supplied binary lesion mask.

Set the `MASK_DIR` environment variable only when you have permission to use the masks and the files are correctly paired with image IDs. The displayed scale bar is measured in **resized image pixels**, not physical millimetres. The code samples up to `N` records from `df_test` and writes figures plus a summary CSV to the configured output directory.


In [ ]:
# ===================== Multi-panel CAM visualization utility =====================
# Features: Grad-CAM and EigenCAM, top-k panels, captions, a colorbar, and a pixel scale bar.
# Optionally reports activation inside/outside a binary mask. Saves and displays each figure.
!pip -q install grad-cam --quiet

import os, cv2, numpy as np, matplotlib.pyplot as plt
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from pytorch_grad_cam import GradCAM, EigenCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from tqdm.auto import tqdm
import pandas as pd

# ----------------- Ayarlar -----------------
OUT_DIR = os.path.join(SUBDIRS["fig"], "cam_pub")
os.makedirs(OUT_DIR, exist_ok=True)
print("Output directory:", OUT_DIR)

# Optional: provide binary masks named with the same image ID (PNG/JPG).
CFG.MASK_DIR = os.environ.get("MASK_DIR", getattr(CFG, "MASK_DIR", None))  # Optional mask directory.

# Match evaluation transforms to the final model's input resolution.
IMG_SZ = getattr(final_model, 'img_size', 672)
eval_tfms = A.Compose([
    A.Resize(IMG_SZ, IMG_SZ),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

# ----------------- Helper functions -----------------
def to_float01(img_tensor):
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    img = (img_tensor.cpu()*std + mean).clamp(0,1).permute(1,2,0).numpy().astype(np.float32)
    return img  # [H,W,3], [0,1]

# ViT/EVA token -> 2D harita
def vit_reshape_transform(tensor):
    if tensor.ndim == 4:   # [B,C,H,W]
        return tensor
    if tensor.ndim != 3:   # Expected shape: [B, N, C].
        raise ValueError(f"Unexpected tensor shape: {tuple(tensor.shape)}")
    B, N, C = tensor.shape
    patch_tokens = tensor[:, 1:, :]               # Remove the class token.
    S = int(np.floor(np.sqrt(patch_tokens.shape[1])))
    patch_tokens = patch_tokens[:, :S*S, :]
    H = W = S
    patch_tokens = patch_tokens.permute(0, 2, 1).contiguous().view(B, C, H, W)
    return patch_tokens

# Hedef katman: norm1 → fallback attn.proj
try:
    TARGET_LAYERS = [getattr(final_model, "blocks")[-1].norm1]
except Exception:
    TARGET_LAYERS = [getattr(getattr(final_model, "blocks")[-1].attn, "proj")]

@torch.no_grad()
def predict(x3chw):
    final_model.eval()
    x = x3chw.unsqueeze(0).to(CFG.DEVICE, non_blocking=True, memory_format=torch.channels_last)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        logits = final_model(x)
        probs  = torch.softmax(logits, 1)
    return probs[0].to(torch.float32).cpu().numpy()  # (C,)

def draw_scale_bar(rgb_uint8, length_px=50, thickness=6, margin=20, text="50 px"):
    canvas = rgb_uint8.copy()
    H, W = canvas.shape[:2]
    x2 = W - margin
    x1 = x2 - length_px
    y  = H - margin
    cv2.line(canvas, (x1, y), (x2, y), (255,255,255), thickness)
    cv2.putText(canvas, text, (x1, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)
    return canvas

def load_mask_if_exists(image_id):
    if not CFG.MASK_DIR or not os.path.isdir(CFG.MASK_DIR):
        return None
    for ext in (".png", ".jpg", ".jpeg", ".bmp", ".tif"):
        p = os.path.join(CFG.MASK_DIR, image_id + ext)
        if os.path.exists(p):
            m = np.array(Image.open(p).convert('L'))
            m = (m > 127).astype(np.uint8)
            return cv2.resize(m, (IMG_SZ, IMG_SZ), interpolation=cv2.INTER_NEAREST)
    return None

def energy_inside_outside(grayscale_cam, mask01):
    # grayscale_cam: [H,W] in [0,1] ; mask01: [H,W] in {0,1}
    if mask01 is None:
        return None, None, None
    g = grayscale_cam.astype(np.float32)
    inside = g[mask01==1]
    outside = g[mask01==0]
    en_in = float(inside.mean()) if inside.size else np.nan
    en_out = float(outside.mean()) if outside.size else np.nan
    ratio = (en_in / (en_out + 1e-8)) if (en_out==en_out) else np.nan
    return en_in, en_out, float(ratio)

def fig_cam_grid(image_id, true_label):
    # ----- 1) Load the image and transform it to a tensor -----
    img_path = os.path.join(CFG.IMAGE_DIR, image_id + ".jpg")
    img_rgb  = np.array(Image.open(img_path).convert("RGB"))
    t = eval_tfms(image=img_rgb)["image"]  # (3,H,W)
    rgb = to_float01(t)                    # float [0,1]
    rgb_u8 = (rgb*255).astype(np.uint8)
    rgb_u8_sb = draw_scale_bar(rgb_u8, length_px=60, text="60 px")

    # ----- 2) Tahmin & top-k -----
    probs = predict(t)                    # (C,)
    topk_idx = probs.argsort()[::-1][:3].tolist()
    pred_idx = topk_idx[0]
    prob_top = [float(probs[i]) for i in topk_idx]

    # ----- 3) CAM'ler -----
    with GradCAM(model=final_model, target_layers=TARGET_LAYERS,
                 reshape_transform=vit_reshape_transform) as gcam, \
         EigenCAM(model=final_model, target_layers=TARGET_LAYERS,
                  reshape_transform=vit_reshape_transform) as ecam:

        xin = t.unsqueeze(0).to(CFG.DEVICE)
        cam_g_top1 = gcam(input_tensor=xin, targets=[ClassifierOutputTarget(pred_idx)])[0]   # [H,W]
        cam_e_top1 = ecam(input_tensor=xin, targets=[ClassifierOutputTarget(pred_idx)])[0]

        # Top-2 & Top-3 (Grad-CAM)
        cam_g_top2 = gcam(input_tensor=xin, targets=[ClassifierOutputTarget(topk_idx[1])])[0] if len(topk_idx)>1 else None
        cam_g_top3 = gcam(input_tensor=xin, targets=[ClassifierOutputTarget(topk_idx[2])])[0] if len(topk_idx)>2 else None

    # ----- 4) Compute activation inside/outside an optional mask -----
    mask = load_mask_if_exists(image_id)
    en_in, en_out, ratio = energy_inside_outside(cam_g_top1, mask)

    # ----- 5) Build a 2 × 3 visualization grid -----
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    fig.suptitle(
        f"{image_id} | pred={CFG.CLASS_NAMES[pred_idx]} ({prob_top[0]:.2f}) | "
        f"true={CFG.CLASS_NAMES[true_label]} | top3={[CFG.CLASS_NAMES[i] for i in topk_idx]}",
        fontsize=12
    )

    # (1) Original image with a pixel scale bar.
    axes[0,0].imshow(rgb_u8_sb); axes[0,0].set_title("Original (+scale bar)"); axes[0,0].axis('off')

    # (2) Grad-CAM top1 overlay
    overlay_g = show_cam_on_image(rgb, cam_g_top1, use_rgb=True)
    axes[0,1].imshow(overlay_g); axes[0,1].set_title(f"Grad-CAM • {CFG.CLASS_NAMES[pred_idx]}"); axes[0,1].axis('off')

    # (3) EigenCAM top1 overlay
    overlay_e = show_cam_on_image(rgb, cam_e_top1, use_rgb=True)
    axes[0,2].imshow(overlay_e); axes[0,2].set_title(f"EigenCAM • {CFG.CLASS_NAMES[pred_idx]}"); axes[0,2].axis('off')

    # (4) Grad-CAM heatmap + colorbar
    im = axes[1,0].imshow(cam_g_top1, cmap='jet', vmin=0, vmax=1); axes[1,0].set_title("Grad-CAM heatmap"); axes[1,0].axis('off')
    cbar = fig.colorbar(im, ax=axes[1,0], fraction=0.046, pad=0.04); cbar.set_label('Activation')

    # (5) Top-2 overlay
    if cam_g_top2 is not None:
        overlay_g2 = show_cam_on_image(rgb, cam_g_top2, use_rgb=True)
        axes[1,1].imshow(overlay_g2); axes[1,1].set_title(f"Top-2 • {CFG.CLASS_NAMES[topk_idx[1]]} ({prob_top[1]:.2f})")
        axes[1,1].axis('off')
    else:
        axes[1,1].axis('off')

    # (6) Top-3 overlay
    if cam_g_top3 is not None:
        overlay_g3 = show_cam_on_image(rgb, cam_g_top3, use_rgb=True)
        axes[1,2].imshow(overlay_g3); axes[1,2].set_title(f"Top-3 • {CFG.CLASS_NAMES[topk_idx[2]]} ({prob_top[2]:.2f})")
        axes[1,2].axis('off')
    else:
        axes[1,2].axis('off')

    # Add mask activation statistics to the caption when a mask is available.
    caption = f"Pred={CFG.CLASS_NAMES[pred_idx]} (p={prob_top[0]:.2f}); True={CFG.CLASS_NAMES[true_label]}; Top3={[(CFG.CLASS_NAMES[i], f'{probs[i]:.2f}') for i in topk_idx]}"
    if ratio is not None:
        caption += f"\nMask energy: inside={en_in:.3f}, outside={en_out:.3f}, inside/outside={ratio:.2f}"
    fig.text(0.5, 0.01, caption, ha='center', fontsize=10)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Kaydet
    out_png = os.path.join(OUT_DIR, f"PUB_CAM_{image_id}.png")
    fig.savefig(out_png, dpi=300)
    plt.show()
    plt.close(fig)

    return {
        "image": image_id,
        "true": CFG.CLASS_NAMES[true_label],
        "pred": CFG.CLASS_NAMES[pred_idx],
        "top1_prob": prob_top[0],
        "top2": CFG.CLASS_NAMES[topk_idx[1]] if len(topk_idx)>1 else None,
        "top2_prob": prob_top[1] if len(topk_idx)>1 else None,
        "top3": CFG.CLASS_NAMES[topk_idx[2]] if len(topk_idx)>2 else None,
        "top3_prob": prob_top[2] if len(topk_idx)>2 else None,
        "mask_inside": en_in,
        "mask_outside": en_out,
        "mask_ratio_in_out": ratio,
        "file": out_png
    }

# ----------------- Generate visualizations for a sample of test images -----------------
N = 25  # Number of test images to visualize.
sample = df_test.sample(n=min(N, len(df_test)), random_state=42).reset_index(drop=True)
rows = []
for _, r in tqdm(sample.iterrows(), total=len(sample)):
    rows.append(fig_cam_grid(r["image"], r["label"]))

# Save a summary table as a CSV file.
df_out = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, "summary_cam_pub.csv")
df_out.to_csv(csv_path, index=False)
print("Finished. Summary CSV:", csv_path)
print("Visualization files:", OUT_DIR)
